In [0]:
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

df = spark.table("workspace.default.players_db").toPandas()
df.head()

,excl_rf,Player,Position,Age,value_m_euro,price_m_euro,pricemvalue_difference_m,arriving_league,arriving_club,departing_league,departing_club,player_id,country_of_citizenship,country_of_birth,foot,height_in_cm,season,release_clause,months_remaining_contract
0,1,Neymar,Attacking Midfield,25,100,222.0,-122.0,LaLiga,Barcelona,Ligue 1,PSG,68290,Brazil,Brazil,Right,175,17/18,1,47
1,2,Kylian Mbappe,Centre-Forward,19,120,180.0,-60.0,Ligue 1,Monaco,Ligue 1,PSG,342229,France,France,Right,178,18/19,0,19
2,3,Ousmane Dembele,Centre-Forward,20,33,148.0,-115.0,Bundesliga,Dortmund,LaLiga,Barcelona,288230,France,France,Both,178,17/18,0,46
3,4,Alexander Isak,Centre-Forward,25,120,145.0,-25.0,Premier League,Newcastle,Premier League,Liverpool,349066,Sweden,Sweden,Right,192,25/26,0,36
4,5,Philippe Coutinho,Attacking Midfield,25,90,135.0,-45.0,Premier League,Liverpool,LaLiga,Barcelona,80444,Brazil,Brazil,Right,172,17/18,0,53


In [0]:
# Fix swapped columns FIRST before anything else
df = df.rename(columns={
    'arriving_league': 'selling_league',
    'departing_league': 'buying_league',
    'arriving_club': 'selling_club',
    'departing_club': 'buying_club'
})

# Clean whitespace after rename
df['selling_league'] = df['selling_league'].str.strip()
df['buying_league'] = df['buying_league'].str.strip()
df['foot'] = df['foot'].str.strip()
df['Position'] = df['Position'].str.strip()

# Remove release clause players
df = df[df['release_clause'] == 0]

# Select features
features = ['Age', 'value_m_euro', 'height_in_cm', 'months_remaining_contract', 
            'Position', 'foot', 'buying_league', 
            'selling_league', 'country_of_citizenship']

# Use .copy() to avoid SettingWithCopyWarning
df_model = df[features + ['price_m_euro']].dropna().copy()

# Encode categoricals
le = LabelEncoder()
for col in ['Position', 'foot', 'buying_league', 'selling_league', 'country_of_citizenship']:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print(df_model.shape)
df_model.head()

(219, 10)


,Age,value_m_euro,height_in_cm,months_remaining_contract,Position,foot,buying_league,selling_league,country_of_citizenship,price_m_euro
1,19,120,178,19,3,2,4,10,16,180.0
2,20,33,178,46,3,0,2,0,16,148.0
3,25,120,192,36,3,2,5,12,37,145.0
4,25,90,172,53,0,2,2,12,4,135.0
6,20,120,186,24,0,2,2,0,15,127.0


In [0]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

# --- MODEL --- no cap, full price range
X = df_model[features]
y = df_model['price_m_euro']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=1000, max_depth=6, min_samples_leaf=2, min_samples_split=5, max_features='sqrt', random_state=42)
gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, min_samples_leaf=3, random_state=42)

ensemble = VotingRegressor(estimators=[('rf', rf), ('gb', gb)])
ensemble.fit(X_train, y_train)

y_pred = ensemble.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

# --- MARKET TREND (CIES validated 9% annual inflation) ---
df['season_start'] = df['season'].str[:2].astype(int) + 2000

annual_inflation = 0.09
last_avg = df[df['season_start'] == df['season_start'].max()]['price_m_euro'].mean()
last_season = df['season_start'].max()

future_seasons_list = []
for i in range(1, 11):
    future_season_start = last_season + i
    projected_fee = last_avg * ((1 + annual_inflation) ** i)
    season_label = f"{str(future_season_start)[2:]}/{str(future_season_start+1)[2:]}"
    future_seasons_list.append({
        'season_label': season_label,
        'avg_fee_predicted': round(projected_fee, 2),
        'type': 'Predicted'
    })

future_seasons = pd.DataFrame(future_seasons_list)

season_avg_all = df.groupby('season_start')['price_m_euro'].mean().reset_index()
season_avg_all['season_label'] = season_avg_all['season_start'].apply(lambda x: f"{str(x)[2:]}/{str(x+1)[2:]}")
season_avg_all['avg_fee_predicted'] = season_avg_all['price_m_euro']
season_avg_all['type'] = 'Actual'

market_trend = pd.concat([
    season_avg_all[['season_label', 'avg_fee_predicted', 'type']],
    future_seasons[['season_label', 'avg_fee_predicted', 'type']]
])

spark.createDataFrame(market_trend).write.mode("overwrite").saveAsTable("workspace.default.market_trend_predictions")
print("Market trend saved!")
print(f"Using CIES validated 9% annual inflation rate")
print(future_seasons[['season_label', 'avg_fee_predicted']])

RMSE: 14.72
R²: 0.2042
Market trend saved!
Using CIES validated 9% annual inflation rate
  season_label  avg_fee_predicted
0        27/28              60.45
1        28/29              65.89
2        29/30              71.82
3        30/31              78.29
4        31/32              85.33
5        32/33              93.01
6        33/34             101.38
7        34/35             110.51
8        35/36             120.45
9        36/37             131.29


In [0]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Save all label encoders WITH their mappings
encoders = {}

# Use fresh copy of original df for encoding
df_encode = df[features + ['price_m_euro']].dropna().copy()

for col in ['Position', 'foot', 'buying_league', 'selling_league', 'country_of_citizenship']:
    enc = LabelEncoder()
    enc.fit(df_encode[col].astype(str))  # FIT only, don't transform yet
    encoders[col] = enc

# Save encoder mappings to tables — these power the dashboard dropdowns
for col, enc in encoders.items():
    mapping_df = pd.DataFrame({
        'column_name': col,
        'label': enc.classes_,
        'encoded_value': range(len(enc.classes_))
    })
    spark.createDataFrame(mapping_df).write.mode("overwrite").saveAsTable(f"workspace.default.encoder_{col}")
    print(f"Saved encoder for {col} — {len(enc.classes_)} unique values")

# Pull fresh display df from original df with readable values
df_display = df[['Player', 'Age', 'Position', 'foot', 'height_in_cm',
                  'country_of_citizenship', 'season', 'buying_league', 
                  'selling_league', 'value_m_euro', 'price_m_euro',
                  'months_remaining_contract']].dropna().copy()

# Create completely separate encoded copy for prediction ONLY
df_pred_only = df_display[features].copy()
for col in ['Position', 'foot', 'buying_league', 'selling_league', 'country_of_citizenship']:
    df_pred_only[col] = encoders[col].transform(df_pred_only[col].astype(str))

# Predict using encoded copy, keep df_display with original readable values
df_display['predicted_price'] = ensemble.predict(df_pred_only[features])
df_display['residual'] = df_display['price_m_euro'] - df_display['predicted_price']
df_display['RMSE'] = rmse
df_display['R2'] = r2

# Save predictions table
spark.createDataFrame(df_display).write.mode("overwrite").saveAsTable("workspace.default.player_predictions")
print("Player predictions saved!")

# Model performance metrics for counter widgets
metrics_df = pd.DataFrame({
    'metric': ['RMSE', 'R2', 'Total_Players', 'Inflation_Rate'],
    'value': [round(rmse, 2), round(r2, 4), len(df_display), 9.0]
})
spark.createDataFrame(metrics_df).write.mode("overwrite").saveAsTable("workspace.default.model_metrics")
print("Model metrics saved!")

# Feature importance for dashboard chart
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': ensemble.estimators_[0].feature_importances_
}).sort_values('Importance', ascending=False)

spark.createDataFrame(importance_df).write.mode("overwrite").saveAsTable("workspace.default.feature_importance")
print("Feature importance saved!")

# Verify predictions look correct
print("\nSample predictions:")
print(df_display[['Player', 'Age', 'Position', 'value_m_euro', 'price_m_euro', 'predicted_price']].head(10))

print("\nAll tables saved:")
print("✓ workspace.default.player_predictions")
print("✓ workspace.default.model_metrics") 
print("✓ workspace.default.feature_importance")
print("✓ workspace.default.market_trend_predictions")
print("✓ workspace.default.encoder_Position")
print("✓ workspace.default.encoder_foot")
print("✓ workspace.default.encoder_buying_league")
print("✓ workspace.default.encoder_selling_league")
print("✓ workspace.default.encoder_country_of_citizenship")

Saved encoder for Position — 10 unique values
Saved encoder for foot — 3 unique values
Saved encoder for buying_league — 10 unique values
Saved encoder for selling_league — 16 unique values
Saved encoder for country_of_citizenship — 43 unique values
Player predictions saved!
Model metrics saved!
Feature importance saved!

Sample predictions:
               Player  Age  ... price_m_euro  predicted_price
1       Kylian Mbappe   19  ...        180.0        68.556211
2     Ousmane Dembele   20  ...        148.0        65.453224
3      Alexander Isak   25  ...        145.0        69.032777
4   Philippe Coutinho   25  ...        135.0        69.557454
6     Jude Bellingham   20  ...        127.0        69.583188
7       Florian Wirtz   22  ...        125.0        68.678481
9         Eden Hazard   28  ...        120.8        67.712431
12  Cristiano Ronaldo   33  ...        117.0        68.602983
13        Declan Rice   24  ...        116.6        68.340820
14     Moises Caicedo   21  ...     

In [0]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Save all label encoders WITH their mappings
encoders = {}

# Use fresh copy of original df for encoding
df_encode = df[features + ['price_m_euro']].dropna().copy()

for col in ['Position', 'foot', 'buying_league', 'selling_league', 'country_of_citizenship']:
    enc = LabelEncoder()
    enc.fit(df_encode[col].astype(str))
    encoders[col] = enc

# Save encoder mappings to tables
for col, enc in encoders.items():
    mapping_df = pd.DataFrame({
        'column_name': col,
        'label': enc.classes_,
        'encoded_value': range(len(enc.classes_))
    })
    spark.createDataFrame(mapping_df).write.mode("overwrite").saveAsTable(f"workspace.default.encoder_{col}")
    print(f"Saved encoder for {col} — {len(enc.classes_)} unique values")

# Pull fresh display df
df_display = df[['Player', 'Age', 'Position', 'foot', 'height_in_cm',
                  'country_of_citizenship', 'season', 'buying_league', 
                  'selling_league', 'value_m_euro', 'price_m_euro',
                  'months_remaining_contract']].dropna().copy()

# Encoded copy for prediction only
df_pred_only = df_display[features].copy()
for col in ['Position', 'foot', 'buying_league', 'selling_league', 'country_of_citizenship']:
    df_pred_only[col] = encoders[col].transform(df_pred_only[col].astype(str))

# Raw model prediction
raw_pred = ensemble.predict(df_pred_only[features])

# Blend model prediction with market value
# Higher value players get weighted more toward their market value
# This fixes the compression issue for high value players
value = df_display['value_m_euro'].values
avg_premium = (df_display['price_m_euro'] / df_display['value_m_euro']).median()
weight = np.clip(value / value.max(), 0.3, 0.7)
blended_pred = (1 - weight) * raw_pred + weight * value * avg_premium

df_display['predicted_price'] = np.round(blended_pred, 2)
df_display['residual'] = df_display['price_m_euro'] - df_display['predicted_price']
df_display['RMSE'] = rmse
df_display['R2'] = r2

# Save predictions table
spark.createDataFrame(df_display).write.mode("overwrite").saveAsTable("workspace.default.player_predictions")
print("Player predictions saved!")

# Model performance metrics
metrics_df = pd.DataFrame({
    'metric': ['RMSE', 'R2', 'Total_Players', 'Inflation_Rate'],
    'value': [round(rmse, 2), round(r2, 4), len(df_display), 9.0]
})
spark.createDataFrame(metrics_df).write.mode("overwrite").saveAsTable("workspace.default.model_metrics")
print("Model metrics saved!")

# Feature importance
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': ensemble.estimators_[0].feature_importances_
}).sort_values('Importance', ascending=False)

spark.createDataFrame(importance_df).write.mode("overwrite").saveAsTable("workspace.default.feature_importance")
print("Feature importance saved!")

# Verify predictions
print("\nSample predictions:")
print(df_display[['Player', 'Age', 'Position', 'value_m_euro', 'price_m_euro', 'predicted_price']].head(10))

print("\nAll tables saved:")
print("✓ workspace.default.player_predictions")
print("✓ workspace.default.model_metrics")
print("✓ workspace.default.feature_importance")
print("✓ workspace.default.market_trend_predictions")
print("✓ workspace.default.encoder_Position")
print("✓ workspace.default.encoder_foot")
print("✓ workspace.default.encoder_buying_league")
print("✓ workspace.default.encoder_selling_league")
print("✓ workspace.default.encoder_country_of_citizenship")

Saved encoder for Position — 10 unique values
Saved encoder for foot — 3 unique values
Saved encoder for buying_league — 10 unique values
Saved encoder for selling_league — 16 unique values
Saved encoder for country_of_citizenship — 43 unique values
Player predictions saved!
Model metrics saved!
Feature importance saved!

Sample predictions:
               Player  Age  ... price_m_euro  predicted_price
1       Kylian Mbappe   19  ...        180.0           151.75
2     Ousmane Dembele   20  ...        148.0            95.58
3      Alexander Isak   25  ...        145.0           145.37
4   Philippe Coutinho   25  ...        135.0           116.39
6     Jude Bellingham   20  ...        127.0           143.36
7       Florian Wirtz   22  ...        125.0           159.84
9         Eden Hazard   28  ...        120.8           167.57
12  Cristiano Ronaldo   33  ...        117.0           121.66
13        Declan Rice   24  ...        116.6           111.93
14     Moises Caicedo   21  ...     

In [0]:
# Check feature importance
importance_check = pd.DataFrame({
    'Feature': features,
    'Importance': ensemble.estimators_[0].feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importances:")
print(importance_check)

# Check correlation
print(f"\nCorrelation value_m_euro vs price_m_euro:")
print(df_display['value_m_euro'].corr(df_display['price_m_euro']))

print(f"\nvalue_m_euro sample:")
print(df_display['value_m_euro'].describe())

Feature Importances:
                     Feature  Importance
1               value_m_euro    0.535839
2               height_in_cm    0.099505
0                        Age    0.086928
3  months_remaining_contract    0.059998
4                   Position    0.052744
8     country_of_citizenship    0.048635
7             selling_league    0.043482
6              buying_league    0.038448
5                       foot    0.034421

Correlation value_m_euro vs price_m_euro:
0.6716142190124726

value_m_euro sample:
count    219.000000
mean      48.899543
std       22.022694
min        0.000000
25%       35.000000
50%       45.000000
75%       60.000000
max      150.000000
Name: value_m_euro, dtype: float64


In [0]:
# Save the full dataset with original readable values for similar player lookup
df_similar = df[['Player', 'Age', 'Position', 'foot', 'height_in_cm',
                  'country_of_citizenship', 'season', 'buying_league',
                  'selling_league', 'value_m_euro', 'price_m_euro',
                  'months_remaining_contract']].dropna().copy()

# Add position group
df_similar['position_group'] = df_similar['Position'].apply(lambda x: 
    'Attacker' if x in ['Centre-Forward', 'Left Winger', 'Right Winger', 'Attacking Midfield']
    else 'Midfielder' if x in ['Central Midfield', 'Defensive Midfield']
    else 'Defender' if x in ['Centre-Back', 'Left-Back', 'Right-Back']
    else 'Goalkeeper'
)

# Add age group
df_similar['age_group'] = df_similar['Age'].apply(lambda x:
    'Teen (< 20)' if x < 20
    else 'Young (20-23)' if x <= 23
    else 'Prime (24-27)' if x <= 27
    else 'Veteran (28+)'
)

# Encoded copy for prediction
df_similar_pred = df_similar[features].copy()
for col in ['Position', 'foot', 'buying_league', 'selling_league', 'country_of_citizenship']:
    df_similar_pred[col] = encoders[col].transform(df_similar_pred[col].astype(str))

# Apply same blending as Cell 4
raw_pred = ensemble.predict(df_similar_pred[features])
value = df_similar['value_m_euro'].values
avg_premium = (df_similar['price_m_euro'] / df_similar['value_m_euro']).median()
weight = np.clip(value / value.max(), 0.3, 0.7)
blended_pred = (1 - weight) * raw_pred + weight * value * avg_premium

df_similar['predicted_price'] = np.round(blended_pred, 2)
df_similar['season_start'] = df_similar['season'].str[:2].astype(int) + 2000

spark.createDataFrame(df_similar).write.mode("overwrite").saveAsTable("workspace.default.similar_players")
print("Similar players table saved!")
print(f"Total players: {len(df_similar)}")
print(df_similar[['Player', 'Position', 'Age', 'price_m_euro', 'predicted_price']].head(10))

Similar players table saved!
Total players: 219
               Player            Position  Age  price_m_euro  predicted_price
1       Kylian Mbappe      Centre-Forward   19         180.0           151.75
2     Ousmane Dembele      Centre-Forward   20         148.0            95.58
3      Alexander Isak      Centre-Forward   25         145.0           145.37
4   Philippe Coutinho  Attacking Midfield   25         135.0           116.39
6     Jude Bellingham  Attacking Midfield   20         127.0           143.36
7       Florian Wirtz  Attacking Midfield   22         125.0           159.84
9         Eden Hazard         Left Winger   28         120.8           167.57
12  Cristiano Ronaldo      Centre-Forward   33         117.0           121.66
13        Declan Rice    Central Midfield   24         116.6           111.93
14     Moises Caicedo  Defensive Midfield   21         116.0            83.66


In [0]:
# Extract coefficients from the model and data
avg_premium = (df['price_m_euro'] / df['value_m_euro']).median()

# Position premium coefficients
position_coef = df.groupby('Position').apply(
    lambda x: (x['price_m_euro'] / x['value_m_euro']).median()
).reset_index()
position_coef.columns = ['Position', 'position_premium']

# Age coefficients
age_coef = df.groupby('Age').apply(
    lambda x: (x['price_m_euro'] / x['value_m_euro']).median()
).reset_index()
age_coef.columns = ['Age', 'age_premium']

# League coefficients
league_coef = df.groupby('buying_league').apply(
    lambda x: (x['price_m_euro'] / x['value_m_euro']).median()
).reset_index()
league_coef.columns = ['buying_league', 'league_premium']

# Foot coefficients
foot_coef = df.groupby('foot').apply(
    lambda x: (x['price_m_euro'] / x['value_m_euro']).median()
).reset_index()
foot_coef.columns = ['foot', 'foot_premium']

# Months remaining — manual bucketing instead of pd.cut
def months_bucket(m):
    if m <= 12: return '0-12'
    elif m <= 24: return '13-24'
    elif m <= 36: return '25-36'
    elif m <= 48: return '37-48'
    else: return '48+'

df['months_bucket'] = df['months_remaining_contract'].apply(months_bucket)
months_coef = df.groupby('months_bucket').apply(
    lambda x: (x['price_m_euro'] / x['value_m_euro']).median()
).reset_index()
months_coef.columns = ['months_bucket', 'months_premium']

# Save all coefficients
spark.createDataFrame(position_coef).write.mode("overwrite").saveAsTable("workspace.default.coef_position")
spark.createDataFrame(age_coef).write.mode("overwrite").saveAsTable("workspace.default.coef_age")
spark.createDataFrame(league_coef).write.mode("overwrite").saveAsTable("workspace.default.coef_league")
spark.createDataFrame(foot_coef).write.mode("overwrite").saveAsTable("workspace.default.coef_foot")
spark.createDataFrame(months_coef).write.mode("overwrite").saveAsTable("workspace.default.coef_months")

global_coef = pd.DataFrame({'metric': ['avg_premium'], 'value': [avg_premium]})
spark.createDataFrame(global_coef).write.mode("overwrite").saveAsTable("workspace.default.coef_global")

print(f"Global avg premium: {avg_premium:.4f}")
print("\nPosition coefficients:")
print(position_coef.sort_values('position_premium', ascending=False))
print("\nLeague coefficients:")
print(league_coef.sort_values('league_premium', ascending=False))
print("\nFoot coefficients:")
print(foot_coef)
print("\nMonths coefficients:")
print(months_coef)

/home/spark-6d4736bc-3762-47d2-b1d3-2a/.ipykernel/2726/command-8370359838711670-3342033715:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  position_coef = df.groupby('Position').apply(
/home/spark-6d4736bc-3762-47d2-b1d3-2a/.ipykernel/2726/command-8370359838711670-3342033715:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  age_coef = df.groupby('Age').apply(
/home/spark-6d4736bc-3762-47d2-b1d3-2a/.ipykern

Global avg premium: 1.2647

Position coefficients:
             Position  position_premium
2         Centre-Back          1.553571
9          Right-Back          1.354286
5          Goalkeeper          1.321310
1    Central Midfield          1.285714
8        Right Winger          1.280000
3      Centre-Forward          1.266667
7           Left-Back          1.258643
6         Left Winger          1.257143
0  Attacking Midfield          1.250000
4  Defensive Midfield          1.200000

League coefficients:
      buying_league  league_premium
3     Liga Portugal        2.950000
9      Super League        1.714286
6  Saudi Pro League        1.516667
5    Premier League        1.280000
2            LaLiga        1.250000
4           Ligue 1        1.199273
7           Serie A        1.181429
1           England        1.128444
8      Stars League        1.125000
0        Bundesliga        0.960390

Foot coefficients:
    foot  foot_premium
0   Both      1.218182
1   Left      1.287302
2 

In [0]:
# Calculate missing coefficients
# Selling league (arriving_league in original = selling_league after rename)
selling_league_coef = df.groupby('selling_league').apply(
    lambda x: (x['price_m_euro'] / x['value_m_euro']).median()
).reset_index()
selling_league_coef.columns = ['selling_league', 'selling_league_premium']

# Nationality
nationality_coef = df.groupby('country_of_citizenship').apply(
    lambda x: (x['price_m_euro'] / x['value_m_euro']).median()
).reset_index()
nationality_coef.columns = ['country_of_citizenship', 'nationality_premium']

# Height buckets
def height_bucket(h):
    if h < 175: return 'Short (< 175cm)'
    elif h <= 184: return 'Medium (175-184cm)'
    elif h <= 190: return 'Tall (185-190cm)'
    else: return 'Very Tall (> 190cm)'

df['height_bucket'] = df['height_in_cm'].apply(height_bucket)
height_coef = df.groupby('height_bucket').apply(
    lambda x: (x['price_m_euro'] / x['value_m_euro']).median()
).reset_index()
height_coef.columns = ['height_bucket', 'height_premium']

# Season
season_coef = df.groupby('season').apply(
    lambda x: (x['price_m_euro'] / x['value_m_euro']).median()
).reset_index()
season_coef.columns = ['season', 'season_premium']

# Buying club
buying_club_coef = df.groupby('buying_club').apply(
    lambda x: (x['price_m_euro'] / x['value_m_euro']).median()
).reset_index()
buying_club_coef.columns = ['buying_club', 'buying_club_premium']

# Selling club
selling_club_coef = df.groupby('selling_club').apply(
    lambda x: (x['price_m_euro'] / x['value_m_euro']).median()
).reset_index()
selling_club_coef.columns = ['selling_club', 'selling_club_premium']

# Add future seasons to season coefficients using CIES 9% inflation
last_season_coef = season_coef['season_premium'].iloc[-1]  # most recent season's premium

future_season_rows = []
last_season_year = 25  # 25/26 is last in dataset

for i in range(1, 12):  # 26/27 through 36/37
    future_year_start = last_season_year + i
    future_year_end = future_year_start + 1
    future_label = f"{str(future_year_start).zfill(2)}/{str(future_year_end).zfill(2)}"
    future_premium = last_season_coef * ((1.09) ** i)
    future_season_rows.append({
        'season': future_label,
        'season_premium': round(future_premium, 4)
    })

future_season_coef = pd.DataFrame(future_season_rows)
season_coef = pd.concat([season_coef, future_season_coef])

print("Season coefficients with future projections:")
print(season_coef)

# Save all
spark.createDataFrame(selling_league_coef).write.mode("overwrite").saveAsTable("workspace.default.coef_selling_league")
spark.createDataFrame(nationality_coef).write.mode("overwrite").saveAsTable("workspace.default.coef_nationality")
spark.createDataFrame(height_coef).write.mode("overwrite").saveAsTable("workspace.default.coef_height")
spark.createDataFrame(season_coef).write.mode("overwrite").saveAsTable("workspace.default.coef_season")
spark.createDataFrame(buying_club_coef).write.mode("overwrite").saveAsTable("workspace.default.coef_buying_club")
spark.createDataFrame(selling_club_coef).write.mode("overwrite").saveAsTable("workspace.default.coef_selling_club")

print("All additional coefficients saved!")
print("\nSelling league coefficients:")
print(selling_league_coef.sort_values('selling_league_premium', ascending=False))
print("\nHeight coefficients:")
print(height_coef.sort_values('height_premium', ascending=False))
print("\nSeason coefficients:")
print(season_coef.sort_values('season', ascending=False))
print("\nNationality coefficients (top 15):")
print(nationality_coef.sort_values('nationality_premium', ascending=False).head(15))

/home/spark-6d4736bc-3762-47d2-b1d3-2a/.ipykernel/2726/command-8370359838711674-1147774383:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  selling_league_coef = df.groupby('selling_league').apply(
/home/spark-6d4736bc-3762-47d2-b1d3-2a/.ipykernel/2726/command-8370359838711674-1147774383:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nationality_coef = df.groupby('country_of_citizenship').apply(
/home/spa

Season coefficients with future projections:
   season  season_premium
0   09 10        1.264706
1   13/14        1.177105
2   14/15        1.571538
3   15/16        1.734444
4   16/17        1.714286
5   17/18        1.672000
6   18/19        1.189167
7   19/20        1.184211
8   20/21        1.294167
9   21/22        1.207540
10  22/23        1.311111
11  23/24        1.331818
12  24/25        1.112500
13  25/26        1.251250
14  26/27        1.128444
0   26/27        1.230000
1   27/28        1.340700
2   28/29        1.461400
3   29/30        1.592900
4   30/31        1.736300
5   31/32        1.892500
6   32/33        2.062800
7   33/34        2.248500
8   34/35        2.450900
9   35/36        2.671400
10  36/37        2.911900
All additional coefficients saved!

Selling league coefficients:
      selling_league  selling_league_premium
3             France                3.180000
8   Liga Profesional                2.950000
9    Liga ZON Sagres                1.956522
1       